# PanDerm 픽셀 단위 영향도 분석: Oral_Diseases Occlusion-Saliency

**목적**: `Oral_Diseases` 개별 테스트 이미지의 **어떤 픽셀 영역**이 Linear Probe 분류 결정을 끌어냈는지 측정합니다.
샘플 단위 LOO([panderm_sample_influence_oral_diseases_20260702.ipynb](panderm_sample_influence_oral_diseases_20260702.ipynb))의 **픽셀 버전**입니다.

**방법**: Occlusion sweep
- frozen PanDerm Large encoder + Linear Probe(baseline) 고정
- 이미지의 한 영역을 가림(occlude) → encoder→probe 재통과 → 확률 하락폭 측정
- `saliency = baseline_prob − occluded_prob` (양수 = 그 영역이 해당 클래스를 **지지**)
- LOO 대응: `influence = baseline − without_sample` ↔ `saliency = baseline − occluded`

**범위 (정직하게)**: 측정 대상은 **개별 예측 확률**이며, 집계 지표(Balanced Accuracy)가 아닙니다.
픽셀×재학습(BACC 영향도)은 비현실적이라 제외합니다 — 자세한 한계는 마지막 셀 참고.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
from pathlib import Path
from sklearn.linear_model import LogisticRegression

# ── 경로 설정 (LOO 노트북과 동일 규약)
NOTEBOOK_DIR  = Path(os.getcwd())
PROJECT_ROOT  = (NOTEBOOK_DIR / "../../..").resolve()
CLASSIFICATION_DIR = PROJECT_ROOT / "PanDerm" / "classification"
if str(CLASSIFICATION_DIR) not in sys.path:
    sys.path.insert(0, str(CLASSIFICATION_DIR))

DATA_ROOT   = PROJECT_ROOT / "PanDerm" / "data"
OUTPUT_ROOT = PROJECT_ROOT / "PanDerm" / "output_dir"
FIGURES_DIR = PROJECT_ROOT / "PanDerm" / "Linear Evaluation" / "figures"
CACHE_DIR   = OUTPUT_ROOT / "oral_diseases_features_cache"
SAL_DIR     = OUTPUT_ROOT / "oral_diseases_panderm_large_lp" / "saliency_maps"
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(SAL_DIR, exist_ok=True)

# ── 데이터셋 설정
DATASET_NAME = "Oral_Diseases"
META_CSV   = DATA_ROOT / "Oral_Diseases" / "Linear Evaluation" / "oral_diseases_multiclass.csv"
IMAGE_ROOT = DATA_ROOT / "Oral_Diseases"
CLASS_NAMES = ["CaS", "CoS", "Gum", "MC", "OC", "OLP", "OT"]
OLP_CLASS_IDX = 5   # OLP = 집중 분석 대상 (Recall 0.30, 최저)
NUM_CLASSES = 7

# ── 모델 설정
MODEL_NAME = "PanDerm_Large_LP"
CHECKPOINT = PROJECT_ROOT / "PanDerm" / "checkpoint" / "panderm_ll_data6_checkpoint-499.pth"

# ── Occlusion 하이퍼파라미터 (정밀 모드: PATCH=32, STRIDE=8)
PATCH  = 16          # ViT 패치 정렬 -> 14x14=196 변형/이미지
STRIDE = 16
OCC_BATCH = 64       # GPU 메모리에 맞춰 조정
MAX_OLP_CASES = 12       # 상세 시각화할 OLP 오분류 상한
MAX_OTHER_CASES = 6      # 기타 클래스 오분류 대표 상한
MAX_AGG_PER_CLASS = 15   # 클래스 집계 시 클래스당 샘플 상한

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print("device:", device, "| checkpoint exists:", CHECKPOINT.exists())

## 1. 모델 & 변환 로드

frozen encoder와 eval transform을 로드합니다. (이 분석은 occlusion 변형 이미지를 매번 통과시켜야 하므로 캐시만으로는 불가, encoder 필수)

In [ ]:
from models import get_encoder
from torchvision import transforms
import argparse

args = argparse.Namespace(model=MODEL_NAME, pretrained_checkpoint=str(CHECKPOINT),
                          batch_size=OCC_BATCH, num_workers=4, nb_classes=NUM_CLASSES, percent_data=1.0)
model, eval_transform = get_encoder(args, MODEL_NAME)   # Resize(256)->CenterCrop(224)->ToTensor->Normalize
model.eval().to(device)

# 표시용 변환 (정규화 제외) — heatmap 아래 깔 RGB 224 이미지
display_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor()
])

@torch.no_grad()
def embed(batch_tensor):
    """[B,3,224,224] 정규화 텐서 -> [B,D] numpy 임베딩 (frozen encoder)."""
    out = model.forward_features(batch_tensor.to(device), is_train=False)
    return out.detach().cpu().numpy()

def load_image(filename):
    """반환: (정규화 텐서[3,224,224], 표시용 텐서[3,224,224]). derm_data 와 동일하게 root+filename 접합."""
    path = str(IMAGE_ROOT) + "/" + filename
    img = Image.open(path).convert("RGB")
    return eval_transform(img), display_transform(img)

print("encoder loaded. feature dim:", embed(torch.zeros(1, 3, 224, 224)).shape)

## 2. 피처 로드 (LOO 캐시 재사용)

baseline probe 학습용 train 피처와, 오분류 식별용 test 피처를 로드합니다. 캐시가 없으면 추출합니다.

In [ ]:
CACHE = {n: CACHE_DIR / f"{n}.npy" for n in
         ["train_feats","train_labels","train_filenames","test_feats","test_labels","test_filenames"]}

if all(p.exists() for p in CACHE.values()):
    print("✅ 캐시된 피처를 로드합니다.")
    train_feats_np  = np.load(CACHE["train_feats"])
    train_labels_np = np.load(CACHE["train_labels"])
    test_feats_np   = np.load(CACHE["test_feats"])
    test_labels_np  = np.load(CACHE["test_labels"])
    test_filenames  = list(np.load(CACHE["test_filenames"], allow_pickle=True))
else:
    print("⏳ 캐시 없음 — PanDerm 모델로 피처를 추출합니다...")
    from datasets.derm_data import Derm_Dataset
    from panderm_model.downstream.extract_features import extract_features_from_dataloader
    df = pd.read_csv(META_CSV)
    IMAGE_ROOT_STR = str(IMAGE_ROOT) + "/"
    def make_loader(split):
        ds = Derm_Dataset(df=df, root=IMAGE_ROOT_STR, transforms=eval_transform,
                          train=(split=="train"), val=(split=="val"), test=(split=="test"), binary=False)
        return torch.utils.data.DataLoader(ds, batch_size=OCC_BATCH, shuffle=False, num_workers=4, pin_memory=True)
    tr = extract_features_from_dataloader(args, model, make_loader("train"))
    te = extract_features_from_dataloader(args, model, make_loader("test"))
    train_feats_np, train_labels_np = tr["embeddings"], tr["labels"]
    test_feats_np,  test_labels_np, test_filenames = te["embeddings"], te["labels"], te["filenames"]
    for k, v in {"train_feats":train_feats_np,"train_labels":train_labels_np,
                 "test_feats":test_feats_np,"test_labels":test_labels_np}.items():
        np.save(CACHE[k], v)
    np.save(CACHE["train_filenames"], np.array(tr["filenames"], dtype=object))
    np.save(CACHE["test_filenames"],  np.array(test_filenames, dtype=object))

print("train:", train_feats_np.shape, "| test:", test_feats_np.shape)
print("test 레이블 분포:", dict(zip(*np.unique(test_labels_np, return_counts=True))))

## 3. Baseline Linear Probe 학습 & 오분류 식별

`linear_probe.py` 와 동일 설정(`C = feat_dim × NUM_C / 100`, LBFGS, max_iter=1000)으로 probe를 **1회** 학습하고, OLP 오분류 샘플을 찾습니다.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, recall_score

NUM_C = len(np.unique(train_labels_np))
C = (train_feats_np.shape[1] * NUM_C) / 100.0   # linear_probe.py 의 use_sklearn 경로와 동일
probe = LogisticRegression(C=C, max_iter=1000, random_state=100)
probe.fit(train_feats_np, train_labels_np)
cls_to_col = {int(c): i for i, c in enumerate(probe.classes_)}

test_probs = probe.predict_proba(test_feats_np)
test_preds = probe.classes_[test_probs.argmax(1)]
bacc = balanced_accuracy_score(test_labels_np, test_preds)
rec  = recall_score(test_labels_np, test_preds, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
print(f"C={C:.2f}  Test Balanced Accuracy={bacc:.4f}")
print("per-class recall:", dict(zip(CLASS_NAMES, rec.round(3))))

mis_mask = test_preds != test_labels_np
olp_mis_idx     = np.where((test_labels_np == OLP_CLASS_IDX) & mis_mask)[0]
olp_correct_idx = np.where((test_labels_np == OLP_CLASS_IDX) & ~mis_mask)[0]
print(f"OLP 오분류 {len(olp_mis_idx)}건, 정분류 {len(olp_correct_idx)}건")

## 4. Occlusion-Saliency 함수

패치를 슬라이드하며 가린 변형들을 미니배치로 한 번에 encoder→probe 통과시켜 확률 하락폭(saliency)을 계산합니다. `baseline=0.0` 은 정규화 공간의 평균(회색) occluder.

In [ ]:
def occlusion_saliency(norm_img, target_class, patch=PATCH, stride=STRIDE, batch=OCC_BATCH, baseline=0.0):
    """norm_img [3,224,224] 정규화 텐서. 반환: (sal_grid [n_i,n_j], base_prob)."""
    H, W = norm_img.shape[1], norm_img.shape[2]
    rows = list(range(0, H - patch + 1, stride))
    cols = list(range(0, W - patch + 1, stride))
    positions = [(i, j) for i in rows for j in cols]
    col = cls_to_col[int(target_class)]
    base_prob = float(probe.predict_proba(embed(norm_img.unsqueeze(0)))[0, col])
    drops = np.zeros(len(positions), dtype=np.float32)
    for s in range(0, len(positions), batch):
        chunk = positions[s:s + batch]
        variants = norm_img.unsqueeze(0).repeat(len(chunk), 1, 1, 1).clone()
        for k, (i, j) in enumerate(chunk):
            variants[k, :, i:i + patch, j:j + patch] = baseline
        probs = probe.predict_proba(embed(variants))[:, col]
        drops[s:s + len(chunk)] = base_prob - probs
    return drops.reshape(len(rows), len(cols)), base_prob

def show_saliency(idx, also_pred=True, save_prefix=None):
    fn = test_filenames[idx]
    true_c, pred_c = int(test_labels_np[idx]), int(test_preds[idx])
    norm_img, disp_img = load_image(fn)
    disp = disp_img.permute(1, 2, 0).numpy()
    panels = [("원본  %s\ntrue=%s | pred=%s" % (Path(fn).name, CLASS_NAMES[true_c], CLASS_NAMES[pred_c]), None)]
    sal_t, bp_t = occlusion_saliency(norm_img, true_c)
    panels.append(("saliency · true=%s (p=%.2f)" % (CLASS_NAMES[true_c], bp_t), sal_t))
    if save_prefix:
        np.save(SAL_DIR / f"{save_prefix}_{Path(fn).stem}_true.npy", sal_t)
    if also_pred and pred_c != true_c:
        sal_p, bp_p = occlusion_saliency(norm_img, pred_c)
        panels.append(("saliency · pred=%s (p=%.2f)" % (CLASS_NAMES[pred_c], bp_p), sal_p))
        if save_prefix:
            np.save(SAL_DIR / f"{save_prefix}_{Path(fn).stem}_pred.npy", sal_p)
    vmax = max(np.abs(s).max() for _, s in panels if s is not None)
    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    axes = np.atleast_1d(axes)
    for ax, (title, sal) in zip(axes, panels):
        ax.imshow(disp)
        if sal is not None:
            ax.imshow(sal, cmap="bwr", vmin=-vmax, vmax=vmax, alpha=0.5,
                      extent=[0, 224, 224, 0], interpolation="bilinear")
        ax.set_title(title, fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()
    return fig

print("red = 해당 클래스를 지지하는 영역(가리면 확률↓), blue = 억제 영역")

## 5. OLP 오분류 픽셀 기여도 (true=OLP vs pred=오답)

각 행: `[원본 | OLP 근거 영역 | 오답 클래스 근거 영역]`. 모델이 OLP를 놓치고 **어느 영역 때문에 다른 클래스로 판단했는지** 확인합니다.

In [ ]:
if len(olp_mis_idx) == 0:
    print("OLP 오분류 샘플이 없습니다.")
for idx in olp_mis_idx[:MAX_OLP_CASES]:
    show_saliency(idx, also_pred=True, save_prefix="olp_mis")

## 6. 정분류 OLP 대조

정상 분류된 OLP의 근거 영역과 비교해, 오분류 케이스에서 어떤 픽셀 단서가 누락/교란되었는지 대조합니다.

In [ ]:
for idx in olp_correct_idx[:4]:
    show_saliency(idx, also_pred=False, save_prefix="olp_correct")

## 7. 기타 클래스 오분류 (대표)

OLP 외 클래스의 오분류도 소수 표본으로 점검합니다. (전수 아님)

In [ ]:
other_mis = np.where(mis_mask & (test_labels_np != OLP_CLASS_IDX))[0]
print(f"기타 클래스 오분류 {len(other_mis)}건 — 대표 {min(MAX_OTHER_CASES, len(other_mis))}건 표시")
for idx in other_mis[:MAX_OTHER_CASES]:
    show_saliency(idx, also_pred=True)

## 8. 클래스별 평균 Occlusion-Saliency (집계)

클래스마다 (target=해당 클래스) saliency를 평균내 **probe가 의존하는 공간 패턴**(중심 vs 주변 등)을 봅니다.

> ⚠️ 이는 개별 **예측 기여도**의 공간 집계이며, **BACC(테스트 성능) 영향도가 아닙니다.**

In [ ]:
agg = {}
for c in range(NUM_CLASSES):
    idxs = np.where(test_labels_np == c)[0][:MAX_AGG_PER_CLASS]
    if len(idxs) == 0:
        continue
    acc = None
    for idx in idxs:
        norm_img, _ = load_image(test_filenames[idx])
        sal, _ = occlusion_saliency(norm_img, c)
        acc = sal if acc is None else acc + sal
    agg[c] = acc / len(idxs)

ncol = 4; nrow = int(np.ceil(len(agg) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 3.2 * nrow))
axes = np.array(axes).reshape(-1)
vmax = max(np.abs(v).max() for v in agg.values())
im = None
for ax, (c, m) in zip(axes, agg.items()):
    im = ax.imshow(m, cmap="bwr", vmin=-vmax, vmax=vmax)
    ax.set_title(f"{CLASS_NAMES[c]} (n≤{MAX_AGG_PER_CLASS})", fontsize=9); ax.axis("off")
for ax in axes[len(agg):]:
    ax.axis("off")
fig.suptitle("클래스별 평균 occlusion saliency (예측 기여도 공간 패턴)", fontsize=11)
if im is not None:
    fig.colorbar(im, ax=list(axes), fraction=0.025)
plt.show()
fig.savefig(FIGURES_DIR / "oral_class_saliency_aggregate.png", bbox_inches="tight")
print("저장:", FIGURES_DIR / "oral_class_saliency_aggregate.png")

## 9. 한계 및 주의

- **예측 기여도 ≠ 성능(BACC) 영향도**: 본 노트북은 개별 예측 확률의 변화만 측정합니다. 픽셀을 섭동한 뒤 probe를 재학습해 집계 지표 변화를 보는 것은 픽셀×재학습 조합이라 비현실적이므로 제외했습니다 (LOO가 샘플 단위에서 멈춘 이유와 동일).
- **occluder 민감도**: saliency는 패치 크기에 민감합니다. 결론 전 `PATCH/STRIDE` 를 16/16 과 32/8 두 설정으로 비교해 robustness를 확인하세요.
- **Segmentation ROI 오버레이 보류**: PanDerm Segmentation 체크포인트는 피부 dermoscopy(ISIC2018/HAM10000)로 학습되어 **Oral_Diseases(구강 점막)에는 out-of-domain** 입니다. 신뢰할 만한 병변 마스크가 나오지 않으므로, saliency를 병변 ROI와 대조하려면 **도메인 일치 분할 모델**을 별도로 사용해야 합니다.
- baseline occluder(회색=정규화 평균)는 데이터 분포 밖 값일 수 있어, blur occluder 등과 교차검증하면 더 견고합니다.
